# 03 — Feature Engineering
Build the training feature matrix combining:
- Exponentially weighted SG rolling averages from the 12 tournaments before each Masters entry
- Augusta-specific career history features
- Augusta fit composite score
- Target columns (finish_position, made_cut, top10, top5, top16, top32, won)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

FEATURES_DIR = Path('../data/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

DECAY = 0.85
WINDOW = 12
SG_COLS = ['sg_total', 'sg_app', 'sg_putt', 'sg_arg', 'sg_ott', 'sg_t2g']

## 1. Load & clean

In [2]:
raw = pd.read_csv('../data/raw/ASA All PGA Raw Data - Tourn Level.csv')
raw = raw.loc[:, ~raw.columns.str.startswith('Unnamed')]

# Clean player names: lowercase + strip
raw['player'] = raw['player'].str.lower().str.strip()
raw['date'] = pd.to_datetime(raw['date'])

# The Kaggle CSV mislabels the November 2020 Masters (postponed from April 2020
# due to COVID) as season=2021 alongside the actual April 2021 Masters.
# Fix: reassign tournament_id 401219478 (played 2020-11-15) to season 2020.
NOV2020_MASTERS_ID = 401219478
raw.loc[raw['tournament id'] == NOV2020_MASTERS_ID, 'season'] = 2020

# Flag Masters rows
raw['is_masters'] = raw['tournament name'].str.contains('Masters', case=False, na=False)

masters_check = raw[raw['is_masters']].groupby('season').size()
print('Masters field sizes by season (after season fix):')
print(masters_check.to_string())
print(f'\nTotal rows: {len(raw):,}')
print(f'Masters rows: {raw["is_masters"].sum()}')
print(f'Non-Masters rows: {(~raw["is_masters"]).sum()}')
print(f'Players: {raw["player"].nunique()}')

Masters field sizes by season (after season fix):
season
2015    79
2016    76
2017    81
2018    79
2019    82
2020    92
2021    87
2022    84

Total rows: 36,864
Masters rows: 660
Non-Masters rows: 36204
Players: 499


## 2. Weighted rolling SG averages (last 12 non-Masters tournaments before each Masters)

In [3]:
masters_df = raw[raw['is_masters']].copy()
non_masters_df = raw[~raw['is_masters']].copy()

# Pre-sort non-Masters by player + date
non_masters_df = non_masters_df.sort_values(['player', 'date'])

def exp_weighted_avg(series, decay):
    """Exponential decay weights: most recent obs = weight 1.0."""
    vals = series.dropna().values
    if len(vals) == 0:
        return np.nan
    # index 0 = oldest, index -1 = most recent
    weights = np.array([decay ** i for i in range(len(vals) - 1, -1, -1)])
    return np.dot(vals, weights) / weights.sum()

records = []
for _, row in masters_df.iterrows():
    player = row['player']
    masters_date = row['date']
    season = row['season']

    prior = non_masters_df[
        (non_masters_df['player'] == player) &
        (non_masters_df['date'] < masters_date)
    ].tail(WINDOW)

    rec = {
        'player': player,
        'season': season,
        'masters_date': masters_date,
        'n_prior_events': len(prior),
    }

    for col in SG_COLS:
        rec[f'{col}_weighted'] = exp_weighted_avg(prior[col], DECAY)

    if len(prior) > 0:
        top10_flags = prior['pos'].apply(lambda p: 1 if pd.notna(p) and p <= 10 else 0)
        rec['top10_rate'] = top10_flags.mean()
        rec['cut_rate'] = prior['made_cut'].mean()
    else:
        rec['top10_rate'] = np.nan
        rec['cut_rate'] = np.nan

    records.append(rec)

form_df = pd.DataFrame(records)
print(f'Form features shape: {form_df.shape}')
print(f'Rows with < {WINDOW} prior events: {(form_df["n_prior_events"] < WINDOW).sum()}')
# Verify no duplicate player-season pairs
dups = form_df.groupby(['player','season']).size()
print(f'Duplicate player-season rows in form_df: {(dups > 1).sum()}')
form_df.head(3)

Form features shape: (660, 12)
Rows with < 12 prior events: 143
Duplicate player-season rows in form_df: 0


,player,season,masters_date,n_prior_events,sg_total_weighted,sg_app_weighted,sg_putt_weighted,sg_arg_weighted,sg_ott_weighted,sg_t2g_weighted,top10_rate,cut_rate
0,abraham ancer,2022,2022-04-10,12,-0.096299,-0.055377,-0.027298,-0.378202,0.277099,-0.157740,0.166667,0.750000
1,adam scott,2022,2022-04-10,12,0.678402,0.481911,0.529326,-0.296182,-0.067498,0.117969,0.250000,0.833333
2,bryson dechambeau,2022,2022-04-10,12,1.402181,0.098737,0.550633,-0.484932,1.234016,0.849416,0.250000,0.750000


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 2.5 — Last-6-events simple averages (same-season, no decay)
# Captures peak/trough short-term form — distinct signal from the 12-event
# exponential-decay averages above.
# ══════════════════════════════════════════════════════════════════════════════

L6_WINDOW = 6
MIN_L6    = 2   # min same-season events to compute; otherwise impute from career

def _l6_stats(rows):
    """Compute l6 feature dict from a date-sorted slice of non-Masters rows."""
    rows = rows.tail(L6_WINDOW)
    top10  = rows['pos'].apply(lambda p: 1.0 if pd.notna(p) and p <= 10 else 0.0)
    finish = rows['pos'].apply(lambda p: float(p) if pd.notna(p) else 80.0)
    return {
        'l6_sg_total_avg': float(rows['sg_total'].mean()),
        'l6_sg_app_avg':   float(rows['sg_app'].mean()),
        'l6_sg_putt_avg':  float(rows['sg_putt'].mean()),
        'l6_top10_rate':   float(top10.mean()),
        'l6_cut_rate':     float(rows['made_cut'].mean()),
        'l6_avg_finish':   float(finish.mean()),
    }

# Career averages (all non-Masters rows) — imputation fallback per player
career_l6 = (
    non_masters_df.sort_values('date')
    .groupby('player')
    .apply(lambda g: pd.Series(_l6_stats(g)))
    .reset_index()
    .set_index('player')
    .to_dict('index')
)

# Global fallback for players absent from non-Masters data
_g_top10  = non_masters_df['pos'].apply(lambda p: 1.0 if pd.notna(p) and p <= 10 else 0.0)
_g_finish = non_masters_df['pos'].apply(lambda p: float(p) if pd.notna(p) else 80.0)
GLOBAL_L6 = {
    'l6_sg_total_avg': float(non_masters_df['sg_total'].mean()),
    'l6_sg_app_avg':   float(non_masters_df['sg_app'].mean()),
    'l6_sg_putt_avg':  float(non_masters_df['sg_putt'].mean()),
    'l6_top10_rate':   float(_g_top10.mean()),
    'l6_cut_rate':     float(non_masters_df['made_cut'].mean()),
    'l6_avg_finish':   float(_g_finish.mean()),
}

l6_records = []
imputed_count = 0

for _, row in masters_df.iterrows():
    player      = row['player']
    masters_date = row['date']
    season      = int(row['season'])

    # Same-season non-Masters events before the Masters date (sorted by date)
    same_season = non_masters_df[
        (non_masters_df['player'] == player) &
        (non_masters_df['season'] == season) &
        (non_masters_df['date']   < masters_date)
    ].sort_values('date')

    rec = {'player': player, 'season': season}

    if len(same_season) >= MIN_L6:
        rec.update(_l6_stats(same_season))
    else:
        # Impute: career averages across all seasons, or global
        ca = career_l6.get(player, GLOBAL_L6)
        rec.update({k: ca.get(k, GLOBAL_L6[k]) for k in GLOBAL_L6})
        imputed_count += 1

    l6_records.append(rec)

l6_df = pd.DataFrame(l6_records)

print(f'l6_df shape: {l6_df.shape}')
print(f'Rows from same-season data (≥{MIN_L6} events): {len(l6_df) - imputed_count}')
print(f'Rows imputed from career averages:              {imputed_count}')
print()
print('l6 feature summary:')
print(l6_df[['l6_sg_total_avg','l6_sg_app_avg','l6_sg_putt_avg',
             'l6_top10_rate','l6_cut_rate','l6_avg_finish']].describe().round(3))

/var/folders/m2/hwnm9hbj2nz1cb_t3z2qxsbh0000gn/T/ipykernel_14966/2444929534.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(_l6_stats(g)))


l6_df shape: (660, 8)
Rows from same-season data (≥2 events): 594
Rows imputed from career averages:              66

l6 feature summary:
       l6_sg_total_avg  l6_sg_app_avg  l6_sg_putt_avg  l6_top10_rate  \
count          640.000        640.000         640.000        660.000   
mean             0.123          0.064          -0.053          0.176   
std              1.267          0.645           0.610          0.199   
min             -6.140         -2.750          -3.293          0.000   
25%             -0.637         -0.305          -0.387          0.000   
50%              0.220          0.074          -0.047          0.167   
75%              0.992          0.500           0.360          0.333   
max              3.310          1.867           1.700          1.000   

       l6_cut_rate  l6_avg_finish  
count      660.000        660.000  
mean         0.691         45.141  
std          0.253         19.021  
min          0.000          3.667  
25%          0.500         31.833

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 2.6 — 3-month SG:T2G features
# External research: 11/13 Masters winners (84.6%) had SG:T2G ≥ 1.7 in the
# 3 months before the Masters.  Source: Kyle Porter (@KylePorterNS)
# ══════════════════════════════════════════════════════════════════════════════

T2G_DAYS      = 90
T2G_THRESHOLD = 1.7
MIN_T2G       = 2   # minimum events in window; otherwise impute from career avg

# Career sg_t2g averages for imputation fallback
career_t2g = (
    non_masters_df.groupby('player')['sg_t2g']
    .mean()
    .to_dict()
)
global_t2g_mean = float(non_masters_df['sg_t2g'].mean())

t2g_records = []
for _, row in masters_df.iterrows():
    player       = row['player']
    masters_date = row['date']
    season       = int(row['season'])

    # Non-Masters events within 90 days before the Masters
    cutoff    = masters_date - pd.Timedelta(days=T2G_DAYS)
    t2g_evts  = non_masters_df[
        (non_masters_df['player'] == player) &
        (non_masters_df['date']   >= cutoff) &
        (non_masters_df['date']   < masters_date)
    ]

    if len(t2g_evts) >= MIN_T2G:
        t2g_avg = float(t2g_evts['sg_t2g'].mean())
    else:
        # Impute: player career average, else global mean
        t2g_avg = career_t2g.get(player, global_t2g_mean)
        if pd.isna(t2g_avg):
            t2g_avg = global_t2g_mean

    t2g_records.append({
        'player':         player,
        'season':         season,
        'sg_t2g_3mo_avg': t2g_avg,
        'sg_t2g_elite':   1 if t2g_avg >= T2G_THRESHOLD else 0,
    })

t2g_df = pd.DataFrame(t2g_records)

n_elite    = (t2g_df['sg_t2g_elite'] == 1).sum()
n_imputed  = sum(
    len(non_masters_df[
        (non_masters_df['player'] == row['player']) &
        (non_masters_df['date']   >= row['date'] - pd.Timedelta(days=T2G_DAYS)) &
        (non_masters_df['date']   < row['date'])
    ]) < MIN_T2G
    for _, row in masters_df.iterrows()
)

print(f't2g_df shape: {t2g_df.shape}')
print(f'Rows with sg_t2g_elite=1 (>=1.7): {n_elite} ({100*n_elite/len(t2g_df):.1f}%)')
print(f'Rows imputed from career avg:      {n_imputed}')
print(f'\nsg_t2g_3mo_avg summary:')
print(t2g_df['sg_t2g_3mo_avg'].describe().round(3))
print(f'\n-- Cross-tab: elite vs actual Masters winner --')
t2g_check = t2g_df.merge(
    masters_df[['player','season','pos']].rename(columns={'pos':'finish'}),
    on=['player','season'], how='left'
)
t2g_check['won'] = (t2g_check['finish'] == 1.0).astype(int)
print(t2g_check.groupby('sg_t2g_elite')['won'].agg(['sum','count','mean']).round(3))

t2g_df shape: (660, 4)
Rows with sg_t2g_elite=1 (>=1.7): 33 (5.0%)
Rows imputed from career avg:      83

sg_t2g_3mo_avg summary:
count    653.000
mean       0.170
std        1.074
min       -4.460
25%       -0.425
50%        0.249
75%        0.889
max        3.405
Name: sg_t2g_3mo_avg, dtype: float64

-- Cross-tab: elite vs actual Masters winner --
              sum  count   mean
sg_t2g_elite                   
0               5    627  0.008
1               3     33  0.091


## 3. Augusta-specific career history features

In [6]:
# Assign numeric finish positions (missed cuts / WD → field_size + 1)
field_size = masters_df.groupby('season')['player'].count()

def assign_finish(row):
    if pd.notna(row['pos']):
        return row['pos']
    return field_size[row['season']] + 1

masters_df = masters_df.copy()
masters_df['finish_pos'] = masters_df.apply(assign_finish, axis=1)
masters_df = masters_df.sort_values(['player', 'date'])

aug_records = []
for _, row in masters_df.iterrows():
    player       = row['player']
    season       = row['season']
    masters_date = row['date']

    # Prior Augusta appearances strictly before this event's date
    prior_aug = masters_df[
        (masters_df['player'] == player) &
        (masters_df['date']   < masters_date)
    ]

    appearances   = len(prior_aug)
    prior_finishes = prior_aug['finish_pos'].dropna()
    avg_finish    = float(prior_finishes.mean()) if appearances > 0 else np.nan
    best_finish   = float(prior_finishes.min())  if appearances > 0 else np.nan

    # masters_wins: count of Augusta wins strictly before this season
    prior_wins    = prior_aug[prior_aug['finish_pos'] == 1.0]
    masters_wins  = len(prior_wins)
    win_seasons   = [int(s) for s in prior_wins['season'].tolist()]
    # recent_form_bonus: 1 if player won Augusta in 2020 or later (before current season)
    recent_form_bonus = 1 if any(s >= 2020 for s in win_seasons) else 0

    aug_records.append({
        'player':              player,
        'season':              season,
        'masters_appearances': appearances,
        'masters_avg_finish':  avg_finish,
        'masters_best_finish': best_finish,
        'masters_wins':        masters_wins,
        'recent_form_bonus':   recent_form_bonus,
    })

aug_df = pd.DataFrame(aug_records)
print(f'Augusta history shape: {aug_df.shape}')
print(f'First-time Masters players:     {(aug_df["masters_appearances"] == 0).sum()}')
print(f'Rows with masters_wins > 0:     {(aug_df["masters_wins"] > 0).sum()}')
print(f'Rows with recent_form_bonus=1:  {(aug_df["recent_form_bonus"] == 1).sum()}')
dups = aug_df.groupby(['player','season']).size()
print(f'Duplicate player-season rows:   {(dups > 1).sum()}')
aug_df.head(3)

Augusta history shape: (660, 7)
First-time Masters players:     213
Rows with masters_wins > 0:     26
Rows with recent_form_bonus=1:  3
Duplicate player-season rows:   0


,player,season,masters_appearances,masters_avg_finish,masters_best_finish,masters_wins,recent_form_bonus
0,aaron wise,2019,0,NaN,NaN,0,0
1,abel gallegos,2020,0,NaN,NaN,0,0
2,abraham ancer,2020,0,NaN,NaN,0,0


## 4. Augusta fit score

In [7]:
feat = form_df.merge(aug_df,  on=['player', 'season'], how='left')
feat = feat.merge(l6_df,   on=['player', 'season'], how='left')
print(f'After merges: {feat.shape}  '
      f'(form={len(form_df)}, aug={len(aug_df)}, l6={len(l6_df)})')

feat['augusta_fit'] = (
    2.0 * feat['sg_app_weighted'] +
    2.0 * feat['sg_putt_weighted'] +
    1.0 * feat['sg_total_weighted'] +
    0.5 * feat['sg_ott_weighted']
) / 5.5

print('\naugusta_fit summary:')
print(feat['augusta_fit'].describe().round(3))

After merges: (660, 23)  (form=660, aug=660, l6=660)

augusta_fit summary:
count    621.000
mean       0.026
std        0.526
min       -2.572
25%       -0.248
50%        0.070
75%        0.383
max        1.610
Name: augusta_fit, dtype: float64


## 5. Add target columns from Masters rows

In [8]:
targets = masters_df[['player', 'season', 'finish_pos', 'made_cut']].copy()

targets['top10'] = (targets['finish_pos'] <= 10).astype(int)
targets['top5']  = (targets['finish_pos'] <= 5).astype(int)
targets['top16'] = (targets['finish_pos'] <= 16).astype(int)
targets['top32'] = (targets['finish_pos'] <= 32).astype(int)
targets['won']   = (targets['finish_pos'] == 1).astype(int)
targets = targets.rename(columns={'finish_pos': 'finish_position'})

print('Target column distributions:')
for col in ['made_cut', 'top32', 'top16', 'top10', 'top5', 'won']:
    n = targets[col].sum()
    pct = 100 * n / len(targets)
    print(f'  {col:15s}: {n:3d} / {len(targets)} ({pct:.1f}%)')

Target column distributions:
  made_cut       : 435 / 660 (65.9%)
  top32          : 262 / 660 (39.7%)
  top16          : 135 / 660 (20.5%)
  top10          :  90 / 660 (13.6%)
  top5           :  49 / 660 (7.4%)
  won            :   8 / 660 (1.2%)


## 6. Merge everything and save

In [9]:
feature_matrix = feat.merge(targets, on=['player', 'season'], how='left')
print(f'Post-merge shape: {feature_matrix.shape}')

# Drop rows with no SG data
pre_drop = len(feature_matrix)
feature_matrix = feature_matrix.dropna(subset=['sg_total_weighted'])
dropped = pre_drop - len(feature_matrix)
if dropped:
    print(f'Dropped {dropped} rows with no prior SG data')

dups = feature_matrix.groupby(['player','season']).size()
if (dups > 1).any():
    print(f'WARNING: {(dups > 1).sum()} duplicate player-season rows remain')
else:
    print('No duplicate player-season rows — clean.')

# Reorder columns
id_cols   = ['player', 'season', 'masters_date', 'n_prior_events']
sg_feat   = [f'{c}_weighted' for c in SG_COLS]
misc_feat = ['top10_rate', 'cut_rate', 'augusta_fit',
             'masters_appearances', 'masters_avg_finish', 'masters_best_finish',
             'masters_wins', 'recent_form_bonus']
l6_feat   = ['l6_sg_total_avg', 'l6_sg_app_avg', 'l6_sg_putt_avg',
             'l6_top10_rate', 'l6_cut_rate', 'l6_avg_finish']
tgt_cols  = ['finish_position', 'made_cut', 'top32', 'top16', 'top10', 'top5', 'won']

feature_matrix = feature_matrix[id_cols + sg_feat + misc_feat + l6_feat + tgt_cols]

out_path = FEATURES_DIR / 'feature_matrix.csv'
feature_matrix.to_csv(out_path, index=False)

print(f'\nFeature matrix shape: {feature_matrix.shape}')
print(f'Columns ({len(feature_matrix.columns)}): {list(feature_matrix.columns)}')
print(f'Saved to: {out_path}')
print()
print('Null counts in new features:')
new_cols = misc_feat[3:] + l6_feat
nulls = feature_matrix[new_cols].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else '  None — clean')
print()
print('First 3 rows:')
feature_matrix.head(3)

Post-merge shape: (660, 31)
Dropped 39 rows with no prior SG data
No duplicate player-season rows — clean.

Feature matrix shape: (621, 31)
Columns (31): ['player', 'season', 'masters_date', 'n_prior_events', 'sg_total_weighted', 'sg_app_weighted', 'sg_putt_weighted', 'sg_arg_weighted', 'sg_ott_weighted', 'sg_t2g_weighted', 'top10_rate', 'cut_rate', 'augusta_fit', 'masters_appearances', 'masters_avg_finish', 'masters_best_finish', 'masters_wins', 'recent_form_bonus', 'l6_sg_total_avg', 'l6_sg_app_avg', 'l6_sg_putt_avg', 'l6_top10_rate', 'l6_cut_rate', 'l6_avg_finish', 'finish_position', 'made_cut', 'top32', 'top16', 'top10', 'top5', 'won']
Saved to: ../data/features/feature_matrix.csv

Null counts in new features:
masters_avg_finish     197
masters_best_finish    197
l6_sg_total_avg          8
l6_sg_app_avg            8
l6_sg_putt_avg           8

First 3 rows:


,player,season,masters_date,n_prior_events,sg_total_weighted,sg_app_weighted,sg_putt_weighted,sg_arg_weighted,sg_ott_weighted,sg_t2g_weighted,...,l6_top10_rate,l6_cut_rate,l6_avg_finish,finish_position,made_cut,top32,top16,top10,top5,won
0,abraham ancer,2022,2022-04-10,12,-0.096299,-0.055377,-0.027298,-0.378202,0.277099,-0.157740,...,0.000000,0.666667,52.500000,85.0,0,0,0,0,0,0
1,adam scott,2022,2022-04-10,12,0.678402,0.481911,0.529326,-0.296182,-0.067498,0.117969,...,0.166667,0.833333,39.833333,48.0,1,0,0,0,0,0
2,bryson dechambeau,2022,2022-04-10,12,1.402181,0.098737,0.550633,-0.484932,1.234016,0.849416,...,0.000000,0.500000,49.750000,85.0,0,0,0,0,0,0


## 7. Missingness audit

In [10]:
print('Null counts per column:')
nulls = feature_matrix.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else '  None — clean dataset')

print(f'\nFinal shape: {feature_matrix.shape[0]} player-year rows × {feature_matrix.shape[1]} columns')
print(f'Seasons: {sorted(feature_matrix["season"].unique())}')
print(f'Unique players: {feature_matrix["player"].nunique()}')

Null counts per column:
masters_avg_finish     197
masters_best_finish    197
l6_sg_total_avg          8
l6_sg_app_avg            8
l6_sg_putt_avg           8

Final shape: 621 player-year rows × 31 columns
Seasons: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Unique players: 202
